In [9]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver
import os
from psycopg import Connection

In [14]:
load_dotenv()

True

In [3]:
llm = ChatGroq(model= "llama-3.1-8b-instant")

In [4]:
def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

In [5]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [15]:
DB_URI = os.getenv("DATABASE_URL")

In [16]:
conn = Connection.connect(
    DB_URI,
    autocommit=True,
    prepare_threshold=0  
)

In [17]:
checkpointer = PostgresSaver(conn)
checkpointer.setup()

In [18]:
graph = builder.compile(checkpointer=checkpointer)

# Thread 1 (remembers)
t1 = {"configurable": {"thread_id": "thread-1"}}
graph.invoke({"messages": [{"role": "user", "content": "Hi, my name is Aryan"}]}, t1)
out1 = graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, t1)
print("Thread-1:", out1["messages"][-1].content)

Thread-1: Your name is Aryan.
